In [3]:
%pip install duckdb
import duckdb

Note: you may need to restart the kernel to use updated packages.


### Question 1: What is the revenue generated by each product category?

In [17]:
result = duckdb.sql("""
    SELECT product_category, 
        COUNT(*) AS num_orders,
        ROUND(SUM(unit_price), 2) AS total_revenue
    FROM coffee_shop.csv
    GROUP BY product_category
    ORDER BY total_revenue DESC;
""")

result

┌────────────────────┬────────────┬───────────────┐
│  product_category  │ num_orders │ total_revenue │
│      varchar       │   int64    │    double     │
├────────────────────┼────────────┼───────────────┤
│ Coffee             │      58416 │      176629.3 │
│ Tea                │      45449 │     128035.35 │
│ Bakery             │      22796 │      80964.14 │
│ Drinking Chocolate │      11468 │      47578.75 │
│ Coffee beans       │       1753 │      36845.25 │
│ Branded            │        747 │       13237.0 │
│ Loose Tea          │       1210 │       11213.6 │
│ Flavours           │       6790 │        5432.0 │
│ Packaged Chocolate │        487 │       4407.64 │
└────────────────────┴────────────┴───────────────┘

### Question 2: What are the 3 most frequently ordered product types in each category?

In [29]:
result = duckdb.sql("""
    WITH temp AS (SELECT product_category, product_type,
        COUNT(*) AS num_orders,
        DENSE_RANK() OVER(PARTITION BY product_category ORDER BY num_orders DESC) AS rank
    FROM coffee_shop.csv
    GROUP BY product_category, product_type
    ORDER BY product_category)
    
    SELECT product_category, product_type, num_orders, rank
    FROM temp
    WHERE rank <= 3;
""")

result

┌────────────────────┬───────────────────────┬────────────┬───────┐
│  product_category  │     product_type      │ num_orders │ rank  │
│      varchar       │        varchar        │   int64    │ int64 │
├────────────────────┼───────────────────────┼────────────┼───────┤
│ Bakery             │ Scone                 │      10173 │     1 │
│ Bakery             │ Pastry                │       6912 │     2 │
│ Bakery             │ Biscotti              │       5711 │     3 │
│ Branded            │ Housewares            │        526 │     1 │
│ Branded            │ Clothing              │        221 │     2 │
│ Coffee             │ Gourmet brewed coffee │      16912 │     1 │
│ Coffee             │ Barista Espresso      │      16403 │     2 │
│ Coffee             │ Organic brewed coffee │       8489 │     3 │
│ Coffee beans       │ Organic Beans         │        415 │     1 │
│ Coffee beans       │ Gourmet Beans         │        366 │     2 │
│ Coffee beans       │ Premium Beans         │  

### Question 3: What is the average bill amount?

In [57]:
result = duckdb.sql("""
    SELECT ROUND(AVG(total_bill), 2) AS avg_bill
    FROM coffee_shop.csv
""")

result

┌──────────┐
│ avg_bill │
│  double  │
├──────────┤
│     4.69 │
└──────────┘

### Question 4: What was the revenue generated in each month in 2023?

In [65]:
result = duckdb.sql("""
    SELECT month, 
        ROUND(SUM(total_bill), 2) AS revenue
    FROM coffee_shop.csv
    WHERE EXTRACT("year" FROM transaction_date) = 2023
    GROUP BY month
    ORDER BY month;
""")

result

┌───────┬───────────┐
│ Month │  revenue  │
│ int64 │  double   │
├───────┼───────────┤
│     1 │  81677.74 │
│     2 │  76145.19 │
│     3 │  98834.68 │
│     4 │ 118941.08 │
│     5 │ 156727.76 │
│     6 │ 166485.88 │
└───────┴───────────┘

### Question 5: How do the results from each month compare to those from the previous month?

In [94]:
result = duckdb.sql("""
    WITH temp1 AS (
        SELECT month,
        ROUND(SUM(total_bill), 2) AS revenue 
        FROM coffee_shop.csv
        GROUP BY month
    ),
    
    temp2 AS (
        SELECT month, revenue,
            LAG(revenue, 1, 0) OVER(ORDER BY month) AS previous_month_revenue
        FROM temp1
    )
    
    SELECT month, revenue, previous_month_revenue,
        ROUND((revenue - previous_month_revenue) / previous_month_revenue, 3) AS change
    FROM temp2;
""")

result

┌───────┬───────────┬────────────────────────┬────────┐
│ Month │  revenue  │ previous_month_revenue │ change │
│ int64 │  double   │         double         │ double │
├───────┼───────────┼────────────────────────┼────────┤
│     1 │  81677.74 │                    0.0 │    inf │
│     2 │  76145.19 │               81677.74 │ -0.068 │
│     3 │  98834.68 │               76145.19 │  0.298 │
│     4 │ 118941.08 │               98834.68 │  0.203 │
│     5 │ 156727.76 │              118941.08 │  0.318 │
│     6 │ 166485.88 │              156727.76 │  0.062 │
└───────┴───────────┴────────────────────────┴────────┘